In [ ]:
import networkx as nx
import pandas as pd
import itertools
from joblib import Parallel, delayed
import numpy as np
from tqdm import tqdm
import community as community_louvain
import warnings
warnings.filterwarnings('ignore')

# Node2Vec imports
from node2vec import Node2Vec
from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity

np.random.seed(42)
print("Bibliotecas carregadas (incluindo Node2Vec)")

In [ ]:
# Carregar grafo
print("Carregando grafo...")
G = nx.read_gml("../data/GraphMissingEdges.gml")

print(f"Nos: {G.number_of_nodes()}")
print(f"Arestas: {G.number_of_edges()}")
print(f"Densidade: {nx.density(G):.6f}")

In [ ]:
# Detectar comunidades
print("Detectando comunidades...")
partition = community_louvain.best_partition(G, random_state=42)
modularity = community_louvain.modularity(partition, G)

communities = {}
for node, comm_id in partition.items():
    if comm_id not in communities:
        communities[comm_id] = []
    communities[comm_id].append(node)

nx.set_node_attributes(G, partition, 'community')

print(f"Comunidades: {len(communities)}")
print(f"Modularidade: {modularity:.4f}")

In [ ]:
# Calcular centralidades
print("Calculando centralidades...")

degreeCentrality = nx.degree_centrality(G)
betCentrality = nx.betweenness_centrality(G)
cloCentrality = nx.closeness_centrality(G)
eigCentrality = nx.eigenvector_centrality(G)
pageRankCentrality = nx.pagerank(G, alpha=0.85)

try:
    katzCentrality = nx.katz_centrality(G, alpha=0.1, beta=1.0)
except:
    katzCentrality = nx.katz_centrality_numpy(G, alpha=0.1, beta=1.0)

sample_nodes = list(G.nodes())[:1000] if len(G.nodes()) > 1000 else list(G.nodes())
loadCentrality_sample = nx.load_centrality(G.subgraph(sample_nodes))
avg_load = np.mean(list(loadCentrality_sample.values()))
loadCentrality = {node: loadCentrality_sample.get(node, avg_load) for node in G.nodes()}

clustering = nx.clustering(G)

print("Centralidades calculadas")

In [ ]:
# Gerar embeddings Node2Vec
print("🚀 INICIANDO GERAÇÃO DE EMBEDDINGS NODE2VEC")
print("=" * 60)

# Informações do grafo
print(f"📊 Estatísticas do grafo:")
print(f"   • Nós: {G.number_of_nodes():,}")
print(f"   • Arestas: {G.number_of_edges():,}")
print(f"   • Densidade: {nx.density(G):.6f}")
print()

# Configurar Node2Vec com parâmetros otimizados
print("⚙️ Configurando parâmetros Node2Vec...")
params = {
    'dimensions': 32,
    'walk_length': 15,
    'num_walks': 50,
    'workers': 4,
    'p': 1,
    'q': 1,
    'seed': 42
}

for key, value in params.items():
    print(f"   • {key}: {value}")
print()

print("🔄 Criando objeto Node2Vec...")
import time
start_time = time.time()

node2vec = Node2Vec(
    G, 
    dimensions=params['dimensions'],
    walk_length=params['walk_length'],
    num_walks=params['num_walks'],
    workers=params['workers'],
    p=params['p'],
    q=params['q'],
    seed=params['seed']
)

setup_time = time.time() - start_time
print(f"✅ Node2Vec configurado em {setup_time:.2f}s")
print()

# Treinar modelo Word2Vec
print("🎯 TREINANDO MODELO WORD2VEC...")
print("─" * 40)

train_params = {
    'window': 10,
    'min_count': 1,
    'batch_words': 4,
    'epochs': 20,
    'sg': 1,
    'hs': 0,
    'negative': 5,
    'workers': 4
}

for key, value in train_params.items():
    print(f"   • {key}: {value}")
print()

print("🔥 Iniciando treinamento...")
train_start = time.time()

model = node2vec.fit(
    window=train_params['window'],
    min_count=train_params['min_count'],
    batch_words=train_params['batch_words'],
    epochs=train_params['epochs'],
    sg=train_params['sg'],
    hs=train_params['hs'],
    negative=train_params['negative'],
    workers=train_params['workers']
)

train_time = time.time() - train_start
print(f"✅ Treinamento concluído em {train_time:.2f}s")
print()

# Extrair embeddings para todos os nós
print("📦 EXTRAINDO EMBEDDINGS...")
print("─" * 30)

embeddings = {}
nodes_list = list(G.nodes())
total_nodes = len(nodes_list)

extract_start = time.time()

# Barra de progresso manual
print("Progresso da extração:")
for i, node in enumerate(nodes_list):
    embeddings[str(node)] = model.wv[str(node)]
    
    # Atualizar progresso a cada 1000 nós ou no final
    if (i + 1) % 1000 == 0 or (i + 1) == total_nodes:
        progress = (i + 1) / total_nodes
        bar_length = 30
        filled_length = int(bar_length * progress)
        bar = '█' * filled_length + '-' * (bar_length - filled_length)
        print(f"[{bar}] {progress:.1%} ({i+1:,}/{total_nodes:,})", end='\r')

extract_time = time.time() - extract_start
print(f"\n✅ Extração concluída em {extract_time:.2f}s")
print()

# Informações dos embeddings
sample_embedding = embeddings[str(nodes_list[0])]
print("📈 INFORMAÇÕES DOS EMBEDDINGS:")
print("─" * 35)
print(f"   • Total de nós: {len(embeddings):,}")
print(f"   • Dimensionalidade: {len(sample_embedding)}")
print(f"   • Tipo de dados: {type(sample_embedding[0])}")
print(f"   • Tamanho em memória: ~{len(embeddings) * len(sample_embedding) * 4 / 1024 / 1024:.1f} MB")
print()

# Estatísticas dos embeddings
all_values = np.concatenate(list(embeddings.values()))
print("📊 ESTATÍSTICAS DOS EMBEDDINGS:")
print("─" * 35)
print(f"   • Média geral: {all_values.mean():.4f}")
print(f"   • Desvio padrão: {all_values.std():.4f}")
print(f"   • Valor mínimo: {all_values.min():.4f}")
print(f"   • Valor máximo: {all_values.max():.4f}")
print()

# Salvar embeddings para reutilização
print("💾 SALVANDO EMBEDDINGS...")
save_start = time.time()
np.save('../pre_processing/node2vec_embeddings_32d.npy', embeddings)
save_time = time.time() - save_start
print(f"✅ Embeddings salvos em {save_time:.2f}s")
print(f"📁 Arquivo: node2vec_embeddings_32d.npy")
print()

# Resumo final
total_time = time.time() - start_time
print("🎉 NODE2VEC CONCLUÍDO COM SUCESSO!")
print("=" * 60)
print(f"⏱️ TEMPO TOTAL: {total_time:.2f}s")
print(f"   • Setup: {setup_time:.2f}s ({setup_time/total_time:.1%})")
print(f"   • Treinamento: {train_time:.2f}s ({train_time/total_time:.1%})")
print(f"   • Extração: {extract_time:.2f}s ({extract_time/total_time:.1%})")
print(f"   • Salvamento: {save_time:.2f}s ({save_time/total_time:.1%})")
print()
print("✨ Embeddings prontos para uso nas features!")

In [ ]:
def extract_features(df, G, communities, degreeCentrality, betCentrality, cloCentrality, 
                    eigCentrality, pageRankCentrality, katzCentrality, loadCentrality, clustering, embeddings):
    """
    Extrai features OTIMIZADAS + Node2Vec embeddings com múltiplas estratégias de combinação
    """
    print(f"Extraindo features OTIMIZADAS + Node2Vec para {len(df)} pares...")
    
    # CENTRALIDADES PRINCIPAIS (top importance)
    df['u_deg'] = df['u'].map(degreeCentrality)
    df['v_deg'] = df['v'].map(degreeCentrality)
    df['u_pagerank'] = df['u'].map(pageRankCentrality)
    df['v_pagerank'] = df['v'].map(pageRankCentrality)
    
    # Features derivadas IMPORTANTES
    df['deg_diff'] = np.abs(df['u_deg'] - df['v_deg'])
    df['pagerank_diff'] = np.abs(df['u_pagerank'] - df['v_pagerank'])
    df['deg_ratio'] = np.minimum(df['u_deg'], df['v_deg']) / (np.maximum(df['u_deg'], df['v_deg']) + 1e-8)
    df['pagerank_ratio'] = np.minimum(df['u_pagerank'], df['v_pagerank']) / (np.maximum(df['u_pagerank'], df['v_pagerank']) + 1e-8)
    
    # APENAS Adamic-Adar (mais importante que Jaccard)
    pairs = list(zip(df["u"], df["v"]))
    chunks = np.array_split(pairs, 8)
    
    def parallel_progress(func, chunks, n_jobs=8):
        results = Parallel(n_jobs=n_jobs, backend='loky')(
            delayed(func)(chunk) for chunk in tqdm(chunks)
        )
        return results
    
    def adar_chunk(chunk):
        return [round(p, 4) for _, _, p in nx.adamic_adar_index(G, chunk)]
    
    results = parallel_progress(adar_chunk, chunks)
    df['aa_score'] = [p for sublist in results for p in sublist]
    
    # FEATURES DE COMUNIDADE (muito importantes)
    df['u_community'] = df['u'].map(lambda node: G.nodes[node]['community'])
    df['v_community'] = df['v'].map(lambda node: G.nodes[node]['community'])
    df['same_community'] = (df['u_community'] == df['v_community']).astype(int)
    
    community_sizes = {comm_id: len(nodes) for comm_id, nodes in communities.items()}
    df['u_comm_size'] = df['u_community'].map(community_sizes)
    df['v_comm_size'] = df['v_community'].map(community_sizes)
    df['comm_size_diff'] = np.abs(df['u_comm_size'] - df['v_comm_size'])
    
    # ===== NODE2VEC EMBEDDINGS FEATURES =====
    print("\n🔮 ADICIONANDO FEATURES NODE2VEC...")
    print("═" * 50)
    
    # Extrair embeddings para cada nó
    print("📦 Extraindo embeddings dos nós...")
    u_embeddings = []
    v_embeddings = []
    
    total_pairs = len(df)
    extract_start = time.time()
    
    for i, (_, row) in enumerate(df.iterrows()):
        u_emb = embeddings[str(row['u'])]
        v_emb = embeddings[str(row['v'])]
        u_embeddings.append(u_emb)
        v_embeddings.append(v_emb)
        
        # Progresso a cada 10k ou no final
        if (i + 1) % 10000 == 0 or (i + 1) == total_pairs:
            progress = (i + 1) / total_pairs
            bar_length = 25
            filled_length = int(bar_length * progress)
            bar = '█' * filled_length + '-' * (bar_length - filled_length)
            elapsed = time.time() - extract_start
            print(f"   [{bar}] {progress:.1%} ({i+1:,}/{total_pairs:,}) - {elapsed:.1f}s", end='\r')
    
    u_embeddings = np.array(u_embeddings)
    v_embeddings = np.array(v_embeddings)
    
    extract_time = time.time() - extract_start
    print(f"\n✅ Embeddings extraídos em {extract_time:.2f}s")
    print(f"📊 Formato: {u_embeddings.shape} para u, {v_embeddings.shape} para v")
    print()
    
    # ESTRATÉGIAS DE COMBINAÇÃO DOS EMBEDDINGS
    print("🎯 APLICANDO ESTRATÉGIAS DE COMBINAÇÃO...")
    print("─" * 45)
    
    # ESTRATÉGIA 1: Produto escalar (dot product) - Mais eficiente
    print("1️⃣ Calculando produto escalar...", end=" ")
    start_step = time.time()
    df['emb_dot_product'] = np.sum(u_embeddings * v_embeddings, axis=1)
    print(f"✅ {time.time() - start_step:.2f}s")
    
    # ESTRATÉGIA 2: Similaridade cosseno - Métrica de similaridade robusta
    print("2️⃣ Calculando similaridade cosseno...", end=" ")
    start_step = time.time()
    emb_cosine_sim = []
    batch_size = 5000  # Processar em lotes para feedback
    
    for i in range(0, len(u_embeddings), batch_size):
        end_idx = min(i + batch_size, len(u_embeddings))
        for j in range(i, end_idx):
            cos_sim = cosine_similarity(u_embeddings[j].reshape(1, -1), v_embeddings[j].reshape(1, -1))[0, 0]
            emb_cosine_sim.append(cos_sim)
        
        # Progresso para cosseno
        if i + batch_size < len(u_embeddings):
            progress = end_idx / len(u_embeddings)
            print(f"\r2️⃣ Calculando similaridade cosseno... {progress:.0%}", end=" ")
    
    df['emb_cosine_sim'] = emb_cosine_sim
    print(f"✅ {time.time() - start_step:.2f}s")
    
    # ESTRATÉGIA 3: Distância euclidiana (invertida para ser similaridade)
    print("3️⃣ Calculando distância euclidiana...", end=" ")
    start_step = time.time()
    emb_euclidean_dist = np.linalg.norm(u_embeddings - v_embeddings, axis=1)
    df['emb_euclidean_sim'] = 1 / (1 + emb_euclidean_dist)
    print(f"✅ {time.time() - start_step:.2f}s")
    
    # ESTRATÉGIA 4: Features estatísticas dos embeddings
    print("4️⃣ Calculando features estatísticas...", end=" ")
    start_step = time.time()
    df['emb_mean_u'] = np.mean(u_embeddings, axis=1)
    df['emb_mean_v'] = np.mean(v_embeddings, axis=1)
    df['emb_std_u'] = np.std(u_embeddings, axis=1)
    df['emb_std_v'] = np.std(v_embeddings, axis=1)
    df['emb_mean_diff'] = np.abs(df['emb_mean_u'] - df['emb_mean_v'])
    df['emb_std_diff'] = np.abs(df['emb_std_u'] - df['emb_std_v'])
    print(f"✅ {time.time() - start_step:.2f}s")
    
    # ESTRATÉGIA 5: Componentes principais dos embeddings (primeiras 5 dimensões mais importantes)
    print("5️⃣ Extraindo componentes principais...", end=" ")
    start_step = time.time()
    for i in range(5):  # Primeiras 5 dimensões
        df[f'emb_u_dim_{i}'] = u_embeddings[:, i]
        df[f'emb_v_dim_{i}'] = v_embeddings[:, i]
        df[f'emb_diff_dim_{i}'] = np.abs(u_embeddings[:, i] - v_embeddings[:, i])
    print(f"✅ {time.time() - start_step:.2f}s")
    print()
    
    # Otimizacao de tipos
    float_cols = ['u_deg', 'v_deg', 'u_pagerank', 'v_pagerank', 'aa_score',
                  'deg_diff', 'pagerank_diff', 'deg_ratio', 'pagerank_ratio',
                  'emb_dot_product', 'emb_cosine_sim', 'emb_euclidean_sim',
                  'emb_mean_u', 'emb_mean_v', 'emb_std_u', 'emb_std_v', 
                  'emb_mean_diff', 'emb_std_diff'] + \
                 [f'emb_u_dim_{i}' for i in range(5)] + \
                 [f'emb_v_dim_{i}' for i in range(5)] + \
                 [f'emb_diff_dim_{i}' for i in range(5)]
    
    int_cols = ['u_community', 'v_community', 'same_community', 
                'u_comm_size', 'v_comm_size', 'comm_size_diff']
    
    df[float_cols] = df[float_cols].astype('float32')
    df[int_cols] = df[int_cols].astype('int16')
    
    # RESUMO FINAL DAS FEATURES NODE2VEC
    print("🎊 RESUMO FINAL DAS FEATURES...")
    print("═" * 50)
    
    node2vec_features = [col for col in df.columns if col.startswith('emb_')]
    classical_features = [col for col in df.columns if not col.startswith('emb_') and col not in ['u', 'v', 'y']]
    
    print(f"📊 FEATURES GERADAS:")
    print(f"   • Clássicas: {len(classical_features)}")
    print(f"   • Node2Vec: {len(node2vec_features)}")
    print(f"   • Total: {len(classical_features) + len(node2vec_features)}")
    print()
    
    print(f"🔮 FEATURES NODE2VEC DETALHADAS:")
    print(f"   • Produto escalar: 1")
    print(f"   • Similaridade cosseno: 1")
    print(f"   • Distância euclidiana: 1")
    print(f"   • Estatísticas (mean/std): 6")
    print(f"   • Componentes principais: 15 (5 dims × 3)")
    print(f"   • Total Node2Vec: {len(node2vec_features)}")
    print()
    
    print(f"💾 INFORMAÇÕES DO DATASET:")
    print(f"   • Shape final: {df.shape}")
    print(f"   • Memória estimada: ~{df.memory_usage(deep=True).sum() / 1024 / 1024:.1f} MB")
    print()
    
    print(f"✨ Features OTIMIZADAS + Node2Vec extraídas com sucesso!")
    return df

print("Funcao de extracao ULTRA-OTIMIZADA + Node2Vec definida!")

## Dataset de Treino (200k samples)

In [ ]:
print("Gerando dataset de treino...")

# Gerar todas as combinacoes
all_edges = list(itertools.combinations(G.nodes(), 2))
df_train = pd.DataFrame(all_edges, columns=['u', 'v'])
print(f"Total de combinacoes: {len(df_train)}")

# Identificar links existentes
edges_set = set(G.edges())
df_train['y'] = [1 if (u, v) in edges_set or (v, u) in edges_set else 0 for u, v in zip(df_train['u'], df_train['v'])]

# Balancear para 200k samples
df_pos = df_train[df_train['y'] == 1]
df_neg = df_train[df_train['y'] == 0]
n_neg_samples = min(200_000 - len(df_pos), len(df_neg))
df_neg_sampled = df_neg.sample(n=n_neg_samples, random_state=42)
df_train_final = pd.concat([df_pos, df_neg_sampled]).reset_index(drop=True)

print(f"Dataset balanceado: {len(df_train_final)} samples")
print(f"Positivos: {df_train_final['y'].sum()}")
print(f"Negativos: {(df_train_final['y'] == 0).sum()}")

In [ ]:
# Extrair features para treino (com Node2Vec)
df_train_processed = extract_features(
    df_train_final, G, communities, degreeCentrality, betCentrality, cloCentrality,
    eigCentrality, pageRankCentrality, katzCentrality, loadCentrality, clustering, embeddings
)

# Salvar (versao ULTRA otimizada + Node2Vec)
output_file = '../data/datasetTreino_node2vec.csv'
df_train_processed.to_csv(output_file, index=False)

print(f"Dataset de treino Node2Vec salvo: {output_file}")
print(f"Shape: {df_train_processed.shape}")
print(f"Features: {df_train_processed.shape[1] - 3} (15 clássicas + Node2Vec)")

## Dataset de Teste

In [ ]:
print("Gerando dataset de teste...")

# Carregar edges para avaliacao
df_test = pd.read_csv('../data/edgesToEvaluate.csv')
df_test_features = df_test.copy()
df_test_features['u'] = df_test_features['venue1']
df_test_features['v'] = df_test_features['venue2']
df_test_features = df_test_features.drop(columns=['linkID', 'venue1', 'venue2'])

print(f"Edges para avaliar: {len(df_test_features)}")

In [ ]:
# Extrair features para teste (com Node2Vec)
df_test_processed = extract_features(
    df_test_features, G, communities, degreeCentrality, betCentrality, cloCentrality,
    eigCentrality, pageRankCentrality, katzCentrality, loadCentrality, clustering, embeddings
)

# Salvar (versao ULTRA otimizada + Node2Vec)
output_file = '../data/datasetTeste_node2vec.csv'
df_test_processed.to_csv(output_file, index=False)

print(f"Dataset de teste Node2Vec salvo: {output_file}")
print(f"Shape: {df_test_processed.shape}")
print(f"Features: {df_test_processed.shape[1] - 2} (15 clássicas + Node2Vec)")

print("VERSÃO ULTRA-OTIMIZADA + NODE2VEC concluida!")
print("=" * 60)
print("Arquivos gerados:")
print(f"- Treino: datasetTreino_node2vec.csv ({df_train_processed.shape})")
print(f"- Teste: datasetTeste_node2vec.csv ({df_test_processed.shape})")
print(f"- Embeddings: node2vec_embeddings_64d.npy")
print("")
print("FEATURES CLÁSSICAS MANTIDAS (15):")
print("- u_deg, v_deg (degree centrality)")
print("- u_pagerank, v_pagerank (PageRank)")  
print("- deg_diff, pagerank_diff, deg_ratio, pagerank_ratio")
print("- aa_score (Adamic-Adar)")
print("- same_community, u_comm_size, v_comm_size, comm_size_diff")
print("")
print("FEATURES NODE2VEC ADICIONADAS:")
print("- emb_dot_product (produto escalar)")
print("- emb_cosine_sim (similaridade cosseno)")
print("- emb_euclidean_sim (distância euclidiana invertida)")
print("- emb_mean_u/v, emb_std_u/v, emb_mean_diff, emb_std_diff")
print("- emb_u/v/diff_dim_0-4 (primeiras 5 dimensões)")
print("")
print("ESTRATÉGIAS DE COMBINAÇÃO IMPLEMENTADAS:")
print("1. Produto escalar dos embeddings")
print("2. Similaridade cosseno")
print("3. Distância euclidiana invertida")
print("4. Features estatísticas (média, desvio)")
print("5. Componentes principais (top 5 dimensões)")
print("")
print(f"Total: ~{df_train_processed.shape[1] - 3} features!")
print("Parâmetros Node2Vec: dim=64, walks=200, length=30")
print("Agora teste no Random Forest com essas features híbridas!")